# 06 Multivariable Analysis: Adjusted Risk Ratios and Logistic Regression

The stratified analysis in Ch05 can only control one confounder at a time. This lesson uses **Modified Poisson regression** to adjust for all factors at once,
computing an **adjusted RR** directly—consistent with Ch03/Ch05. It also runs logistic regression as a comparison,
showing how the OR overestimates the effect at a high attack rate.

Workflow: **data preparation → Crude RR & OR → Adjusted RR (Modified Poisson) → Adjusted OR (Logistic) → comparison → Forest Plot → model diagnostics**

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || True
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# === Step 1: Data preparation + variable recoding ===

import pathlib
import pandas as pd
import numpy as np
import statsmodels.api as sm               # GLM (Modified Poisson)
import statsmodels.formula.api as smf       # formula API (logistic)
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings

# -- CJK font setup (avoid Chinese labels rendering as boxes) --
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

# --- Load the data ---
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")

# --- Build the binary outcome variable ---
# clinical_severity != 'not_ill' means infected (includes mild/moderate/severe)
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

# --- smoking_history three categories → two categories ---
df["ever_smoker"] = (df["smoking_history"] != "never").astype(int)

# --- Convert functional status to an ordered numeric score ---
# bedridden=0 < assisted=1 < independent=2
fs_map = {"bedridden": 0, "assisted": 1, "independent": 2}
df["functional_score"] = df["functional_status"].map(fs_map)

# --- Quick check of the attack rate ---
ar = df["infected"].mean()
print(f"Total: {len(df)} people, infected: {df['infected'].sum()} people")
print(f"Attack rate = {ar:.1%}")
print(f"→ An attack rate of {ar:.0%} is far above 10%, so the OR will markedly overestimate the effect; rely on RR")

## Why use Modified Poisson to compute the RR?

- Ch03: cohort study → effect measure is **RR**
- Ch05: stratified analysis → **MH adjusted RR**
- This chapter: multivariable analysis → **adjusted RR** (Modified Poisson)

**Modified Poisson (Zou 2004)**: a Poisson GLM + robust sandwich SE, where the coefficient = log(RR).
It's like borrowing a hat from a friend (Poisson is meant for count data): the size isn't quite right, but slap on a correction sticker (robust SE) and it fits perfectly.

It also runs **logistic regression** (→ OR), so you can see how much the OR overestimates at a high attack rate.

In [ ]:
# === Step 2: Univariable analysis — Crude RR vs Crude OR ===
# Run Modified Poisson (RR) and logistic (OR) at the same time to see the difference for each variable

from epi_learning.metrics import risk_ratio  # the 2×2 hand-computed RR from Ch03

factors = [
    "shower_use", "hydrotherapy_use", "ever_smoker",
    "comorbidity_chf", "comorbidity_dm", "comorbidity_cancer",
    "comorbidity_copd", "immunosuppressed",
    "age", "functional_score",
]

crude_results = []

for var in factors:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        # (A) Modified Poisson → crude RR
        try:
            mod_p = smf.glm(
                f"infected ~ {var}", data=df,
                family=sm.families.Poisson(),
            ).fit(cov_type="HC0", disp=0)
            rr = np.exp(mod_p.params[var])
            rr_ci = np.exp(mod_p.conf_int().loc[var])
        except Exception:
            continue

        # (B) Logistic → crude OR
        try:
            mod_l = smf.logit(f"infected ~ {var}", data=df).fit(disp=0, method="lbfgs")
            if not mod_l.mle_retvals["converged"]:
                print(f"⚠ {var}: logistic did not converge, skipping")
                continue
            or_val = np.exp(mod_l.params[var])
            or_ci = np.exp(mod_l.conf_int().loc[var])
        except Exception:
            continue

    # (C) 2×2 hand-computed RR for cross-validation (binary variables only)
    hand_rr = ""
    if df[var].dropna().isin([0, 1]).all():
        a = ((df[var] == 1) & (df["infected"] == 1)).sum()
        b = ((df[var] == 1) & (df["infected"] == 0)).sum()
        c = ((df[var] == 0) & (df["infected"] == 1)).sum()
        d = ((df[var] == 0) & (df["infected"] == 0)).sum()
        hand_rr = f"{risk_ratio(a, a+b, c, c+d):.3f}"

    crude_results.append({
        "variable": var,
        "crude_RR": round(rr, 3),
        "RR 95% CI": f"{rr_ci[0]:.3f}–{rr_ci[1]:.3f}",
        "crude_OR": round(or_val, 3),
        "OR 95% CI": f"{or_ci[0]:.3f}–{or_ci[1]:.3f}",
        "hand_RR": hand_rr,
    })

crude_df = pd.DataFrame(crude_results)
print("=== Crude RR vs Crude OR (univariable) ===")
print(crude_df.to_string(index=False))
print()
print("💡 crude_OR is generally larger than crude_RR — the OR overestimation at high attack rates")
print("   The hand_RR column = the 2×2 hand-computed result from Ch03; it should match crude_RR almost exactly")

### Reading the formula syntax

statsmodels borrows the **formula syntax** from the R language to describe "which variables predict the outcome" in a single line:

| Symbol | Meaning | Example |
|------|------|------|
| `~` | "is predicted by" | `infected ~ age` → use age to predict infected |
| `+` | "plus" | `~ age + sex` → put both age and sex into the model |
| `C()` | "treat as categorical" | `C(floor)` → split floor into dummy variables (dummy coding), one 0/1 indicator per floor |

In plain language, `infected ~ shower_use + age + C(floor)` says "use shower use, age, and floor to predict infection." The model automatically adds an intercept term (Intercept), so you don't need to write it.

### Which variables go into the model?—starting from the Ch03 and Ch05 results

A multivariable model isn't about throwing in every column; there needs to be a reason for each one. Reviewing the findings from earlier chapters, we split the variables into four groups:

| Group | Variables | Role | Reason for inclusion |
|------|------|------|----------|
| **Exposure factors** | `shower_use`, `hydrotherapy_use` | Study focus | The significant risk factors screened in Ch03—the question we most want to answer: "are the shower and hydrotherapy the source of infection?" |
| **Host factors** | `age`, `immunosuppressed`, `functional_score` | Potential confounders | `age` = a factor epidemiology routinely adjusts for; `immunosuppressed` = one of the factors with the highest crude RR in Ch03; `functional_score` = the confounder already confirmed in Ch05 |
| **Comorbidities** | `comorbidity_chf/dm/cancer/copd` | Potential confounders | The candidate factors screened in Ch03, put into the full model to see whether the exposure factors' RR changes after adjustment |
| **Location** | `C(floor)` | Potential confounder | Different floors may have different plumbing systems or exposure opportunities, so we need to control for floor differences |

> **Why not include `ever_smoker`?** In the Ch03 univariable screening, smoking had a crude RR close to 1 and did not reach statistical significance. On top of that, it is highly correlated with several comorbidities (collinearity), so including it in the model would only increase the instability of the estimates. That's why we leave it out of the multivariable model.

In [ ]:
# === Step 3: Modified Poisson — multivariable Adjusted RR ===
# The main analysis of the chapter: Poisson GLM + robust SE → coefficient = log(RR)

# --- Formula explanation ---
# infected ~ : use the variables on the right to predict "infected or not"
# shower_use + hydrotherapy_use : exposure factors (study focus)
# age + immunosuppressed + functional_score : host factors (potential confounders)
# comorbidity_chf/dm/cancer/copd : comorbidities (potential confounders)
# C(floor) : floor as a categorical variable (control for location differences)
formula = (
    "infected ~ shower_use + hydrotherapy_use + age + "
    "comorbidity_chf + comorbidity_dm + comorbidity_cancer + "
    "comorbidity_copd + immunosuppressed + functional_score + "
    "C(floor)"     # C(floor) = treat floor as a categorical variable (dummy coding)
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    model_poisson = smf.glm(
        formula, data=df,
        family=sm.families.Poisson(),   # the Poisson shell
    ).fit(cov_type="HC0")               # robust (sandwich) SE

# --- Format into Table 2 layout ---
adj_rr_results = []
for var in model_poisson.params.index:
    if var == "Intercept":
        continue
    coef = model_poisson.params[var]
    ci = model_poisson.conf_int().loc[var]
    adj_rr_results.append({
        "variable": var,
        "adjusted_RR": round(np.exp(coef), 3),
        "95% CI": f"{np.exp(ci[0]):.3f}–{np.exp(ci[1]):.3f}",
        "p-value": round(model_poisson.pvalues[var], 4),
    })

adj_rr_df = pd.DataFrame(adj_rr_results)
print("=== Adjusted RR (Modified Poisson, Table 2) ===")
print(adj_rr_df.to_string(index=False))

In [ ]:
# === Step 4: Logistic Regression — multivariable Adjusted OR (comparison) ===
# Same formula, switch to logistic regression to see how much the OR overestimates relative to the RR

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    model_logit = smf.logit(formula, data=df).fit(disp=0, method="lbfgs")

adj_or_results = []
for var in model_logit.params.index:
    if var == "Intercept":
        continue
    coef = model_logit.params[var]
    ci = model_logit.conf_int().loc[var]
    adj_or_results.append({
        "variable": var,
        "adjusted_OR": round(np.exp(coef), 3),
        "95% CI": f"{np.exp(ci[0]):.3f}–{np.exp(ci[1]):.3f}",
        "p-value": round(model_logit.pvalues[var], 4),
    })

adj_or_df = pd.DataFrame(adj_or_results)
print("=== Adjusted OR (Logistic Regression, Table 2) ===")
print(adj_or_df.to_string(index=False))

In [ ]:
# === Step 5: Crude RR vs Adjusted RR vs Adjusted OR side by side ===

key_vars = ["shower_use", "hydrotherapy_use", "age",
            "comorbidity_chf", "immunosuppressed", "functional_score"]

comparison = []
for var in key_vars:
    c_row = crude_df[crude_df["variable"] == var]
    a_rr_row = adj_rr_df[adj_rr_df["variable"] == var]
    a_or_row = adj_or_df[adj_or_df["variable"] == var]
    if len(c_row) == 0 or len(a_rr_row) == 0 or len(a_or_row) == 0:
        continue
    c_rr = c_row.iloc[0]["crude_RR"]
    a_rr = a_rr_row.iloc[0]["adjusted_RR"]
    a_or = a_or_row.iloc[0]["adjusted_OR"]
    rr_chg = ((a_rr - c_rr) / c_rr * 100) if c_rr != 0 else 0
    or_vs = ((a_or - a_rr) / a_rr * 100) if a_rr != 0 else 0
    comparison.append({
        "variable": var,
        "crude_RR": c_rr,
        "adj_RR": a_rr,
        "adj_OR": a_or,
        "crude→adj RR": f"{rr_chg:+.1f}%",
        "adj RR→OR": f"{or_vs:+.1f}%",
    })

comp_df = pd.DataFrame(comparison)
print("=== Crude RR → Adjusted RR → Adjusted OR comparison ===")
print(comp_df.to_string(index=False))
print()
print("📊 crude→adj RR: the change in RR after controlling for confounders")
print("   adj RR→OR: how much the OR overestimates relative to the RR in the same model")

In [ ]:
# === Step 6: Forest Plot — Adjusted RR ===

plot_vars = [r for r in adj_rr_df["variable"] if not r.startswith("C(floor)")]
plot_data = adj_rr_df[adj_rr_df["variable"].isin(plot_vars)].copy()
plot_data["ci_lo"] = plot_data["95% CI"].str.split("–").str[0].astype(float)
plot_data["ci_hi"] = plot_data["95% CI"].str.split("–").str[1].astype(float)

fig, ax = plt.subplots(figsize=(8, 5))
y_pos = range(len(plot_data))
ax.errorbar(
    plot_data["adjusted_RR"], y_pos,
    xerr=[plot_data["adjusted_RR"] - plot_data["ci_lo"],
          plot_data["ci_hi"] - plot_data["adjusted_RR"]],
    fmt="o", color="#D97757", ecolor="#6A9BCC",
    elinewidth=2, capsize=4, markersize=7,
)
ax.axvline(x=1, color="#6B6B6B", linestyle="--", linewidth=1, label="RR = 1")
ax.set_yticks(list(y_pos))
ax.set_yticklabels(plot_data["variable"])
ax.set_xlabel("Adjusted RR (95% CI)")
ax.set_title("Forest Plot — Adjusted Risk Ratio (Modified Poisson)")
ax.legend(loc="lower right", fontsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# === Step 7: Model diagnostics — AIC comparison ===
# Use AIC to compare the "full model" and the "reduced model" and judge whether we put in too many variables.

# --- Reduced model ---
# Remove the comorbidities that were non-significant or had small effect sizes in the Ch03 screening, plus floor.
# Keep: the core exposure factors (shower_use, hydrotherapy_use)
#       + the theoretically most important adjustment factors (age, immunosuppressed, functional_score)
formula_reduced = (
    "infected ~ shower_use + hydrotherapy_use + age + "
    "immunosuppressed + functional_score"
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    model_reduced = smf.glm(
        formula_reduced, data=df,
        family=sm.families.Poisson(),
    ).fit(cov_type="HC0")

print("=== Model comparison (Modified Poisson) ===")
print(f"  Full model AIC = {model_poisson.aic:.1f}")
print(f"  Reduced model AIC = {model_reduced.aic:.1f}")
print()
if model_reduced.aic < model_poisson.aic:
    print("📉 The reduced model has a smaller AIC → it strikes a better balance")
else:
    print("📈 The full model has a smaller AIC → the extra variables really do contribute")

### How do we choose variables? A comparison of three strategies

Above we only compared the two models "full vs reduced." But in practice, which variables should go into the model? There are three common strategies:

| Strategy | How it works | Pros | Cons |
|------|------|------|------|
| **Forward (add in)** | Start from the empty model and each step add the variable that lowers AIC the most | Simple and intuitive | Easily misses joint effects; the order of addition affects the result |
| **Backward (drop out)** | Start from the full model and each step remove the variable with the smallest effect on AIC | Can see the joint effect of all variables | Needs a large enough sample to include all variables |
| **Change-in-estimate** | Remove candidate confounders one at a time and see whether the exposure factor's RR changes by ≥ 10% | **The epidemiological gold standard**—judges based on "does it confound the exposure effect" | You must define the "exposure factor" first |

**Epidemiology recommends change-in-estimate**, not stepwise (automatic variable selection). The reason is simple: the purpose of our multivariable analysis is to **correctly estimate the effect of the exposure factor**, not to make predictions. Even if a variable's p-value is not significant, as long as it is a confounder (removing it changes the RR by ≥ 10%), it should stay in the model.

Below we implement the change-in-estimate method in Python:

In [ ]:
# === Step 7b: Change-in-Estimate variable selection ===
# The standard epidemiological approach: remove candidate confounders one at a time
# and see how much the exposure factors' (shower_use, hydrotherapy_use) adjusted RR changes.
# A change of ≥ 10% → that variable is a confounder and must stay in the model.

# --- The exposure factors' RR from the full model (baseline) ---
full_rr = {
    var: np.exp(model_poisson.params[var])
    for var in ["shower_use", "hydrotherapy_use"]
}
print("Exposure factors' RR from the full model (baseline):")
for var, rr in full_rr.items():
    print(f"  {var}: {rr:.3f}")
print()

# --- Candidate confounders: test by removing each one ---
confounders = [
    "age", "comorbidity_chf", "comorbidity_dm", "comorbidity_cancer",
    "comorbidity_copd", "immunosuppressed", "functional_score", "C(floor)",
]

cie_results = []
for drop_var in confounders:
    # Build the formula with one variable removed
    keep = [c for c in confounders if c != drop_var]
    formula_test = "infected ~ shower_use + hydrotherapy_use + " + " + ".join(keep)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        m = smf.glm(formula_test, data=df, family=sm.families.Poisson()).fit(
            cov_type="HC0", disp=0
        )

    for exposure in ["shower_use", "hydrotherapy_use"]:
        rr_without = np.exp(m.params[exposure])
        pct_change = (rr_without - full_rr[exposure]) / full_rr[exposure] * 100
        cie_results.append({
            "removed variable": drop_var,
            "exposure factor": exposure,
            "RR after removal": round(rr_without, 3),
            "RR change %": f"{pct_change:+.1f}%",
            "confounder?": "✓ confounder" if abs(pct_change) >= 10 else "",
        })

cie_df = pd.DataFrame(cie_results)
print("=== Change-in-Estimate analysis ===")
print("(After removing a variable, if an exposure factor's RR changes by ≥ 10% → that variable is a confounder)\n")
print(cie_df.to_string(index=False))
print()
print("📋 Look at the 'RR change %' column: variables with a change ≥ 10% are confounders,")
print("   and must stay in the model even if their own p-value is not significant.")

## Summary

| Step | Method | Output |
|------|------|------|
| Crude RR | Univariable Modified Poisson | crude RR |
| Adjusted RR | Multivariable Modified Poisson | **adjusted RR** (main method) |
| Adjusted OR | Multivariable Logistic Regression | adjusted OR (comparison) |
| Side-by-side comparison | Three-column table | crude→adj confounding effect + magnitude of OR overestimation |
| Forest plot | matplotlib errorbar | visualize the adjusted RR |
| Model diagnostics | AIC comparison | full vs reduced |

**Conclusion**: Modified Poisson is the first-choice method for multivariable analysis in a cohort study (it computes the RR directly).
Logistic regression overestimates the effect at a high attack rate (OR > RR), but it remains the standard method for case-control studies or rare diseases.

Next chapter (Ch07): your supervisor wants to know "how many new cases will there be next week?" → time series forecasting